# The Dimensionality Bridge: Classical vs Quantum Visualization

This notebook compares how classical models and quantum feature maps represent the same Iris dataset.

Objectives:
- Train classical baselines (Logistic Regression and RBF-SVM).
- Visualize 2D PCA decision boundaries.
- Build Bell and GHZ circuits as a hardware stress test (not a QML accuracy booster).
- Use a `ZZFeatureMap` to encode 4D Iris vectors in a quantum circuit.
- Run SamplerV2 with PUB inputs on ISA-transpiled circuits.


## Quantum Feature Mapping and the Curse of Dimensionality

Classical models operate in Euclidean feature space and often rely on hand-crafted kernels. As dimensionality grows, distance concentration and sparse sampling make many classical intuitions weaker.

Quantum feature mapping embeds vectors into amplitudes and phases of Hilbert space. The `ZZFeatureMap` introduces pairwise interactions that act like a structured, data-dependent kernel in a larger state space.

Iris itself is not a quantum-advantage dataset. It is a compact testbed for the encoding workflow needed before moving to harder, higher-dimensional tasks.


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_histogram

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

ASSETS_DIR = Path("assets")
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)


## 1) Classical Baselines on Iris (4D Feature Space)


In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "RBF-SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE)),
    ]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"{name} accuracy: {accuracy_score(y_test, preds):.4f}")
    print(classification_report(y_test, preds, target_names=iris.target_names))


## 2) 2D PCA Decision Boundaries

To compare geometric bias, we project the 4D space to two principal components and fit the same model families in that reduced view.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
X_all_pca = np.vstack([X_train_pca, X_test_pca])
y_all = np.concatenate([y_train, y_test])

boundary_models = {
    "Logistic Regression (PCA-2D)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "RBF-SVM (PCA-2D)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE)),
    ]),
}


def plot_decision_surface(ax, model, X_plot, y_plot, title):
    x_min, x_max = X_plot[:, 0].min() - 0.8, X_plot[:, 0].max() + 0.8
    y_min, y_max = X_plot[:, 1].min() - 0.8, X_plot[:, 1].max() + 0.8

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 350),
        np.linspace(y_min, y_max, 350),
    )

    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, zz, alpha=0.30, cmap="coolwarm")
    scatter = ax.scatter(
        X_plot[:, 0],
        X_plot[:, 1],
        c=y_plot,
        cmap="coolwarm",
        edgecolor="black",
        linewidth=0.5,
        s=36,
    )

    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    return scatter


fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for ax, (name, model) in zip(axes, boundary_models.items()):
    model.fit(X_train_pca, y_train)
    pca_preds = model.predict(X_test_pca)
    print(f"{name} test accuracy in PCA-2D: {accuracy_score(y_test, pca_preds):.4f}")
    scatter = plot_decision_surface(ax, model, X_all_pca, y_all, name)

handles, labels = scatter.legend_elements()
axes[-1].legend(handles, iris.target_names, title="Class", loc="best")

classical_boundary_path = ASSETS_DIR / "classical_boundaries.png"
fig.savefig(classical_boundary_path, dpi=220, bbox_inches="tight")
print(f"Saved: {classical_boundary_path}")
plt.show()


## 3) Quantum Feature Map (Iris 4D to Circuit Encoding)

`ZZFeatureMap` applies single-qubit phase rotations and pairwise ZZ interactions so each 4D Iris vector is lifted into entangled quantum features.


In [ ]:
zz_feature_map = ZZFeatureMap(feature_dimension=4, reps=2, entanglement="linear")

feature_map_fig = zz_feature_map.decompose().draw(output="mpl", fold=120)
quantum_feature_path = ASSETS_DIR / "quantum_feature_map.png"
feature_map_fig.savefig(quantum_feature_path, dpi=220, bbox_inches="tight")
print(f"Saved: {quantum_feature_path}")
display(feature_map_fig)


## 4) Entanglement Stress Test: Bell State and GHZ-127

A Bell circuit demonstrates two-qubit maximal entanglement.

For scale, we explicitly construct a 127-qubit GHZ circuit. The saved diagram uses a sparse visual proxy to keep the plot readable while preserving the full GHZ circuit object for execution. This section is a decoherence/noise benchmark and is independent from Iris classification quality.


In [ ]:
def build_bell_state() -> QuantumCircuit:
    qc = QuantumCircuit(2, 2, name="Bell")
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    return qc


def build_ghz_state(num_qubits: int) -> QuantumCircuit:
    qc = QuantumCircuit(num_qubits, num_qubits, name=f"GHZ_{num_qubits}")
    qc.h(0)
    for q in range(num_qubits - 1):
        qc.cx(q, q + 1)
    qc.measure(range(num_qubits), range(num_qubits))
    return qc


def build_sparse_ghz_visual(total_qubits: int, left: int = 4, right: int = 4) -> QuantumCircuit:
    visible = left + right
    qc = QuantumCircuit(visible, visible, name=f"GHZ_{total_qubits}_sparse")

    qc.h(0)
    for idx in range(left - 1):
        qc.cx(idx, idx + 1)

    qc.barrier()
    # Represents the omitted CX chain through hidden middle qubits.
    qc.cx(left - 1, left)
    qc.barrier()

    for idx in range(left, visible - 1):
        qc.cx(idx, idx + 1)

    qc.measure(range(visible), range(visible))
    return qc


bell_circuit = build_bell_state()
ghz_127_circuit = build_ghz_state(127)
ghz_sparse_visual = build_sparse_ghz_visual(127)

bell_fig = bell_circuit.draw(output="mpl")
ghz_sparse_fig = ghz_sparse_visual.draw(output="mpl", fold=80)
ghz_sparse_path = ASSETS_DIR / "ghz_127_circuit.png"
ghz_sparse_fig.savefig(ghz_sparse_path, dpi=220, bbox_inches="tight")

print(f"Saved: {ghz_sparse_path}")
print(
    f"Full GHZ circuit created with {ghz_127_circuit.num_qubits} qubits and "
    f"{ghz_127_circuit.size()} instructions."
)
display(bell_fig)
display(ghz_sparse_fig)


## 5) ISA Circuits + SamplerV2 PUBs

We transpile to an ISA backend target first, then execute with SamplerV2 on a local Aer backend for reproducible simulation.


In [ ]:
USE_REAL_IBM_BACKEND = os.getenv("USE_REAL_IBM_BACKEND", "0") == "1"
IBM_BACKEND_NAME = os.getenv("IBM_BACKEND_NAME", "ibm_sherbrooke")


def get_fake_or_generic_backend():
    try:
        from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

        return FakeSherbrooke()
    except Exception:
        # Fallback for environments without FakeSherbrooke in fake_provider.
        from qiskit.providers.fake_provider import GenericBackendV2

        coupling_map = [(i, i + 1) for i in range(126)] + [(i + 1, i) for i in range(126)]
        return GenericBackendV2(
            num_qubits=127,
            basis_gates=["id", "rz", "sx", "x", "cx"],
            coupling_map=coupling_map,
            seed=RANDOM_STATE,
        )


def build_runtime_service():
    token = os.getenv("IBM_QUANTUM_TOKEN")
    instance = os.getenv("IBM_QUANTUM_INSTANCE")
    channels = ("ibm_quantum_platform", "ibm_cloud")
    last_error = None

    for channel in channels:
        try:
            kwargs = {"channel": channel}
            if token:
                kwargs["token"] = token
            if instance:
                kwargs["instance"] = instance
            return QiskitRuntimeService(**kwargs), channel
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        "Unable to authenticate to IBM Runtime. Set IBM_QUANTUM_TOKEN to a valid key "
        "(and IBM_QUANTUM_INSTANCE if required), or run "
        "QiskitRuntimeService.save_account(...) once."
    ) from last_error


if USE_REAL_IBM_BACKEND:
    runtime_service, runtime_channel = build_runtime_service()
    isa_backend = runtime_service.backend(IBM_BACKEND_NAME)
    sampler_backend = isa_backend
    sampler_mode_label = f"real backend ({isa_backend.name}, channel={runtime_channel})"
else:
    isa_backend = get_fake_or_generic_backend()
    sampler_backend = AerSimulator(method="matrix_product_state")
    sampler_mode_label = "local AerSimulator(method=matrix_product_state)"

pass_manager = generate_preset_pass_manager(optimization_level=3, backend=isa_backend)

feature_map_measured = zz_feature_map.copy()
feature_map_measured.measure_all()

isa_bell = pass_manager.run(bell_circuit)
isa_ghz_127 = pass_manager.run(ghz_127_circuit)
isa_feature_map = pass_manager.run(feature_map_measured)

print(f"ISA backend: {isa_backend.name}")
print(f"Sampler mode: {sampler_mode_label}")
print(f"ISA Bell depth: {isa_bell.depth()}")
print(f"ISA GHZ-127 depth: {isa_ghz_127.depth()}")
print(f"ISA ZZFeatureMap depth: {isa_feature_map.depth()}")


In [ ]:
sampler = Sampler(mode=sampler_backend)


def extract_counts(pub_result):
    data = pub_result.data

    for key in ("meas", "c", "cr"):
        if hasattr(data, key):
            reg = getattr(data, key)
            if hasattr(reg, "get_counts"):
                return reg.get_counts()

    for key in dir(data):
        if key.startswith("_"):
            continue
        reg = getattr(data, key)
        if hasattr(reg, "get_counts"):
            return reg.get_counts()

    raise RuntimeError("No classical register with counts found in SamplerV2 result.")


def normalize_state_key(state, width: int) -> str:
    if isinstance(state, str):
        return state
    if isinstance(state, int):
        return format(state, f"0{width}b")
    return str(state)


def abbreviate_bitstring(bitstring: str, keep: int = 8) -> str:
    if len(bitstring) <= 2 * keep:
        return bitstring
    return f"{bitstring[:keep]}...{bitstring[-keep:]}"


# SamplerV2 PUB tuple for GHZ (unparameterized).
ghz_job = sampler.run([(isa_ghz_127, [])], shots=2048)
ghz_pub_result = ghz_job.result()[0]
ghz_counts_raw = extract_counts(ghz_pub_result)

state_width = isa_ghz_127.num_clbits
ghz_counts = {
    normalize_state_key(state, state_width): count for state, count in ghz_counts_raw.items()
}

total_shots = sum(ghz_counts.values())
ghz_probabilities = {bitstring: count / total_shots for bitstring, count in ghz_counts.items()}
ghz_probabilities_short = {
    abbreviate_bitstring(bitstring): prob for bitstring, prob in ghz_probabilities.items()
}

hist_fig = plot_histogram(ghz_probabilities_short, title="GHZ-127 Quasiprobability (SamplerV2)")
display(hist_fig)

# SamplerV2 PUB tuple for parameterized feature map.
sample_angles = X_train[0].tolist()
feature_job = sampler.run([(isa_feature_map, [sample_angles])], shots=512)
feature_pub_result = feature_job.result()[0]
feature_counts_raw = extract_counts(feature_pub_result)

feature_width = isa_feature_map.num_clbits
feature_counts = {
    normalize_state_key(state, feature_width): count for state, count in feature_counts_raw.items()
}

print("Top GHZ quasiprobability outcomes:")
for state, prob in sorted(ghz_probabilities.items(), key=lambda item: item[1], reverse=True)[:5]:
    print(f"  {abbreviate_bitstring(state)} : {prob:.4f}")

print("\nTop ZZFeatureMap sampled outcomes:")
for state, count in sorted(feature_counts.items(), key=lambda item: item[1], reverse=True)[:5]:
    print(f"  {abbreviate_bitstring(state)} : {count}")


## 6) Interpretation

- Classical PCA plots show directly interpretable low-dimensional class geometry.
- `ZZFeatureMap` demonstrates how 4D vectors become phase-encoded, entangled quantum states.
- Bell and GHZ-127 circuits are a hardware noise stress test; for GHZ-127, the practical outcome on noisy targets is typically signal collapse.
- ISA transpilation + SamplerV2 PUB execution keeps this workflow aligned with modern Qiskit Runtime practices.
